In [1]:
import scanpy as sc
import Spectra as spc
import pandas as pd
import numpy as np
from Spectra import Spectra_util as spc_tl
from Spectra import K_est as kst
from Spectra import default_gene_sets
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/Spectra/Spectra_util.py:3: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


### data & annotations preprocessing

In [2]:
annotations = spc.default_gene_sets.load()

In [3]:
# Load your data
adata = sc.read_h5ad('patient_data/anal_pc5_c21_S1.filtered.h5ad')

In [4]:
metadata = pd.read_csv('data/anal_cancer_n21_meta.csv', low_memory=False)
metadata.head()

,Unnamed: 0,samplename,viral_status,cluster_annot,copykat,pathology_status,cluster_annot2,cluster_annot3
0,S1_AAACAAGCAACAGCACACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,T_cells,not.defined,cancer,CD4+Tcells,Tregs
1,S1_AAACAAGCAATTGAGTACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,Myeloid_cells,diploid,cancer,DC_CLEC9A,DC_CLEC9A
2,S1_AAACAAGCACGTAAATACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,Fibroblast_stromal,not.defined,cancer,CD8+Tcells,Fibroblasts/Stromal
3,S1_AAACAAGCACTATCACACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,B_cells,not.defined,cancer,Plasma_XBP1,Plasma_XBP1
4,S1_AAACAAGCAGAATGAAACTTTAGG_anal_34449_1_1,S1_Sanal_34449,HPV-_HIV+,Endothelials,diploid,cancer,Endothelials,Endothelials


In [5]:
metadata = metadata.set_index('Unnamed: 0')
adata.obs['cluster_annot3'] = metadata.loc[adata.obs_names, 'cluster_annot3']

In [6]:
print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 14133 × 17781
    obs: 'orig.ident', 'cluster_annot3'
    obsm: 'X_harmony', 'X_pca', 'X_umap'
                                           orig.ident       cluster_annot3
S1_AAACAAGCAACAGCACACTTTAGG_anal_34449_1_1         S1                Tregs
S1_AAACAAGCAATTGAGTACTTTAGG_anal_34449_1_1         S1            DC_CLEC9A
S1_AAACAAGCACGTAAATACTTTAGG_anal_34449_1_1         S1  Fibroblasts/Stromal
S1_AAACAAGCACTATCACACTTTAGG_anal_34449_1_1         S1          Plasma_XBP1
S1_AAACAAGCAGAATGAAACTTTAGG_anal_34449_1_1         S1         Endothelials


In [7]:
# Check what cell types are in the default annotations
print("Default annotation keys:", annotations.keys())

Default annotation keys: dict_keys(['B_GC', 'B_memory', 'B_naive', 'CD4_T', 'CD8_T', 'DC', 'ILC3', 'MDC', 'NK', 'Treg', 'gdT', 'mast', 'pDC', 'plasma', 'global'])


In [8]:
# Check what cell types are in the data
print("Cell types in your data:")
print(metadata.cluster_annot3.value_counts())

Cell types in your data:
cluster_annot3
Tumor/Epithelials      171927
Fibroblasts/Stromal     33617
Plasma_XBP1             30276
Endothelials            13279
Mac_C1Q_ZEB2            11520
CD8+Tcells               8802
CD4+Tcells               7574
Tregs                    4528
Bcells_CD19              3986
Mac_C1Q_SPP1             2586
NKcells                  2509
Mon_VCAN                 2489
Mast                     2463
DC_CD207                 2376
CD8+Tcells_Ki67          1868
Neutrophils              1818
DC_LAMP3                 1412
DC_CLEC9A                 838
pDCs                      766
Name: count, dtype: int64


In [9]:
# Map cell types to Spectra's expected cell type labels
annotation_mapping = {
    # T cells
    'CD4+Tcells': 'CD4_T',
    'CD8+Tcells': 'CD8_T',
    'CD8+Tcells_Ki67': 'CD8_T',  # proliferating CD8s still map to CD8_T
    'Tregs': 'Treg',
    'NKcells': 'NK',
    
    # B cells and plasma
    'Bcells_CD19': 'B_naive',  # or B_memory - depends on your data
    'Plasma_XBP1': 'plasma',
    
    # Dendritic cells
    'DC_CD207': 'DC',      # Langerhans/cDC2
    'DC_LAMP3': 'DC',      # mature/migratory DCs
    'DC_CLEC9A': 'DC',     # cDC1
    'pDCs': 'pDC',
    
    # Myeloid
    'Mac_C1Q_ZEB2': 'MDC',  # resident macrophages
    'Mac_C1Q_SPP1': 'MDC',  # inflammatory/SPP1+ macrophages
    'Mon_VCAN': 'MDC',      # classical monocytes
    'Neutrophils': 'MDC',   # if you need to include them
    
    # Other immune
    'Mast': 'mast',
    
    # Non-immune cell types - map to custom labels (not 'global'!)
    'Tumor/Epithelials': 'Epithelial',
    'Fibroblasts/Stromal': 'Fibroblast',
    'Endothelials': 'Endothelial',
}

In [10]:
# Apply the mapping to create the spectra_annot_celltype column
metadata['spectra_annot_celltype'] = metadata['cluster_annot3'].map(annotation_mapping)
# Add to adata
adata.obs['spectra_annot_celltype'] = metadata.loc[adata.obs_names, 'spectra_annot_celltype']

# Check for any unmapped cell types (will be NaN)
print("Unique mapped cell types in adata:")
print(adata.obs['spectra_annot_celltype'].value_counts(dropna=False))

Unique mapped cell types in adata:
spectra_annot_celltype
Epithelial     7299
Fibroblast     1621
plasma         1329
B_naive        1326
CD4_T           624
CD8_T           551
Endothelial     414
MDC             335
Treg            300
DC              136
NK               97
mast             92
pDC               9
Name: count, dtype: int64


In [11]:
# Build annotations_custom with ONLY the cell types present in the data

# Get the unique cell types actually in data (excluding NaN)
adata_celltypes = set(adata.obs['spectra_annot_celltype'].dropna().unique())
print("Cell types in your adata:", adata_celltypes)

# only include 'global' and cell types that exist in the data
annotations_custom = {'global': annotations['global']}
for ct in adata_celltypes:
    if ct in annotations:
        # This cell type has gene sets in the default annotations
        annotations_custom[ct] = annotations[ct]
    else:
        # This cell type (Epithelial, Fibroblast, Endothelial) has no gene sets
        # Add an empty dict so Spectra knows about it
        annotations_custom[ct] = {}

print("\nAnnotations_custom keys:", annotations_custom.keys())

Cell types in your adata: {'MDC', 'Fibroblast', 'NK', 'B_naive', 'Epithelial', 'CD4_T', 'mast', 'DC', 'Endothelial', 'Treg', 'plasma', 'pDC', 'CD8_T'}

Annotations_custom keys: dict_keys(['global', 'MDC', 'Fibroblast', 'NK', 'B_naive', 'Epithelial', 'CD4_T', 'mast', 'DC', 'Endothelial', 'Treg', 'plasma', 'pDC', 'CD8_T'])


In [12]:
# Check if already normalized (values should be small, typically 0-10 range)
print("Max value before normalization:", adata.X.max())

# If max is large, data is raw counts - normalize it
if adata.X.max() > 50:
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print("Max value after normalization:", adata.X.max())
else:
    print("Data appears to already be normalized")

Max value before normalization: 1552
Max value after normalization: 9.023591


In [13]:
annotations_filtered = spc_tl.check_gene_set_dictionary(
    adata,
    annotations_custom,
    obs_key='spectra_annot_celltype',
    global_key='global'
)
print("Gene set dictionary check passed!")

Cell type labels in gene set annotation dictionary and AnnData object are identical
Your gene set annotation dictionary is now correctly formatted.
Gene set dictionary check passed!


In [14]:
# Compute highly variable genes 
sc.pp.highly_variable_genes(adata, n_top_genes=5000)
print(f"Number of highly variable genes: {adata.var['highly_variable'].sum()}")

Number of highly variable genes: 5000


In [15]:
# FIX: Monkey-patch pandas Series to add nonzero() method
# This fixes compatibility between newer pandas and scipy sparse indexing
# The issue is that Spectra's code creates boolean Series that scipy can't handle

if not hasattr(pd.Series, 'nonzero'):
    def _series_nonzero(self):
        return self.to_numpy().nonzero()
    pd.Series.nonzero = _series_nonzero
    print("Applied pandas Series.nonzero() patch")

Applied pandas Series.nonzero() patch


### run & fit spectra model

In [ ]:
# Fit the Spectra model
model = spc.est_spectra(
    adata=adata, 
    gene_set_dictionary=annotations_filtered,  # use the filtered annotations
    use_highly_variable=True,
    cell_type_key="spectra_annot_celltype", 
    use_weights=True,
    lam=0.1,  # varies depending on data and gene sets, try between 0.5 and 0.001
    delta=0.001, 
    kappa=None,
    rho=0.001, 
    use_cell_types=True,
    n_top_vals=50,
    label_factors=True, 
    overlap_threshold=0.2,
    clean_gs=True, 
    min_gs_num=3,
    num_epochs=2000  # use 10000 for real runs
)

Cell type labels in gene set annotation dictionary and AnnData object are identical
Your gene set annotation dictionary is now correctly formatted.


 26%|██▋       | 528/2000 [18:53:45<52:40:46, 128.84s/it] 


KeyboardInterrupt: 

In [ ]:
# View results stored in adata
print("Cell scores shape:", adata.obsm['SPECTRA_cell_scores'].shape)
print("\nFactors stored in adata.uns['SPECTRA_factors']")
print("Markers stored in adata.uns['SPECTRA_markers']")

### interpreting first level of spectra outputs

In [ ]:
print(adata.uns['SPECTRA_L'])

In [ ]:
# === USAGE MATRIX (Cells × Factors) ===
# Get factor labels for nice column names
factor_labels = adata.uns['SPECTRA_overlap'].index.tolist()

usage_matrix = pd.DataFrame(
    adata.obsm['SPECTRA_cell_scores'],
    index=adata.obs_names,
    columns=factor_labels
)
print(f"Usage matrix shape: {usage_matrix.shape}")

# === GEP MATRIX (Factors × Genes) ===
# Get the vocabulary (genes used in model)
vocab_genes = adata.var[adata.var['spectra_vocab']].index.tolist()

gep_matrix = pd.DataFrame(
    adata.uns['SPECTRA_factors'],
    index=factor_labels,
    columns=vocab_genes
)
print(f"GEP matrix shape: {gep_matrix.shape}")

In [ ]:
usage_matrix.head()

In [ ]:
gep_matrix.head()

In [ ]:
print(adata.uns['SPECTRA_markers'][0])  # top genes for factor x

### splitting global factors into cell type specific factors

1) Extract global factors and create cell-type factor profiles

In [ ]:
# Get the number of global factors
n_global_factors = adata.uns['SPECTRA_L']['global']
print(f"Number of global factors: {n_global_factors}")

# Extract cell scores for global factors only (first n_global factors)
cell_scores_global = adata.obsm['SPECTRA_cell_scores'][:, :n_global_factors]

# Get factor labels from the overlap index
factor_labels = adata.uns['SPECTRA_overlap'].index[:n_global_factors].tolist()

# Create a DataFrame with cell scores
cell_scores_df = pd.DataFrame(
    cell_scores_global,
    index=adata.obs_names,
    columns=factor_labels
)

# Add cell type annotation
cell_scores_df['cell_type'] = adata.obs['spectra_annot_celltype'].values

# Remove any cells with missing cell type
cell_scores_df = cell_scores_df.dropna(subset=['cell_type'])

print(f"Cell scores shape: {cell_scores_df.shape}")
print(f"Cell types: {cell_scores_df['cell_type'].unique()}")

2. Calculate mean factor activity per cell type

In [ ]:
# Group by cell type and calculate mean
celltype_factor_mean = cell_scores_df.groupby('cell_type').mean()

# Also calculate median (more robust to outliers)
celltype_factor_median = cell_scores_df.groupby('cell_type').median()

# Calculate standard deviation
celltype_factor_std = cell_scores_df.groupby('cell_type').std()
# Calculate coefficient of variation (std/mean) - measures relative variability
celltype_factor_cv = celltype_factor_std / (celltype_factor_mean + 1e-10)

print(f"\nCell type × Factor matrix shape: {celltype_factor_mean.shape}")

3. Z-score normalize across cell types for better comparison

In [ ]:
# Z-score normalize each factor across cell types
# This shows which cell types have above/below average activity for each factor
celltype_factor_zscore = celltype_factor_median.apply(stats.zscore, axis=0)
# Replace any NaN (from constant columns) with 0
celltype_factor_zscore = celltype_factor_zscore.fillna(0)

4. Identify cell-type specific factors

In [ ]:
def identify_specific_factors(zscore_df, threshold=1.5):
    """
    Identify factors that are specific to one or few cell types.
    
    A factor is considered cell-type specific if:
    - It has z-score > threshold in only 1-2 cell types
    - OR it has z-score < -threshold in most cell types except 1-2
    
    Returns a DataFrame with specificity annotations.
    """
    results = []
    
    for factor in zscore_df.columns:
        scores = zscore_df[factor]
        
        # Cell types with high activity
        high_activity = scores[scores > threshold].index.tolist()
        
        # Cell types with low activity
        low_activity = scores[scores < -threshold].index.tolist()
        
        # Determine specificity
        n_celltypes = len(zscore_df)
        n_high = len(high_activity)
        n_low = len(low_activity)
        
        if n_high == 1:
            specificity = 'highly_specific'
            specific_to = high_activity
        elif n_high == 2:
            specificity = 'moderately_specific'
            specific_to = high_activity
        elif n_high >= n_celltypes - 2 or n_low <= 1:
            specificity = 'broadly_expressed'
            specific_to = []
        else:
            specificity = 'subset_specific'
            specific_to = high_activity
        
        # Get the max z-score and which cell type
        max_zscore = scores.max()
        max_celltype = scores.idxmax()
        
        results.append({
            'factor': factor,
            'specificity': specificity,
            'specific_to': specific_to,
            'n_high_celltypes': n_high,
            'max_zscore': max_zscore,
            'max_celltype': max_celltype,
            'high_celltypes': high_activity,
            'low_celltypes': low_activity
        })
    
    return pd.DataFrame(results)


In [ ]:
# Identify specific factors
factor_specificity = identify_specific_factors(celltype_factor_zscore, threshold=1.5)

print("\n" + "="*60)
print("FACTOR SPECIFICITY SUMMARY")
print("="*60)
print(factor_specificity['specificity'].value_counts())

#  highly specific factors
highly_specific = factor_specificity[factor_specificity['specificity'] == 'highly_specific']

In [ ]:
# Clean up factor names for easier reading
def clean_factor_name(name):
    """Extract just the gene set name from the full factor label."""
    parts = name.split('-X-')
    if len(parts) >= 3:
        return parts[2]  # The gene set name
    return name

factor_specificity['factor_short'] = factor_specificity['factor'].apply(clean_factor_name)

# Create a cell type × factor heatmap-ready matrix
# with factor specificity annotations
celltype_factor_annotated = celltype_factor_zscore.copy()

### visualizations

1. Heatmap of cell type × factor activity

**Color Interpretation:**
- **Deep RED** (positive z-score): Factor is highly active in that cell type relative to others
- **Deep BLUE** (negative z-score): Factor is suppressed in that cell type relative to others
- **WHITE/pale** (z ≈ 0): Average activity across cell types

**Pattern Recognition:**
- **Horizontal patterns (rows)**: A row with one red cell and many blue cells indicates a factor highly specific to a single cell type
- **Vertical patterns (columns)**: Columns predominantly one color indicate cell types with globally elevated or suppressed factor activity
- **Original ordering**: Preserves factor indexing, which may group biologically related pathways if your annotations are organized

**Biological Insights:**
- Identify which pathways are enriched in specific cell types
- Spot factors that distinguish tumor-infiltrating immune cells from stromal components
- See if certain metabolic or signaling pathways dominate particular cell populations

In [ ]:
# transpose so factors are rows, cell types are columns
heatmap_data_full = celltype_factor_zscore.T.copy()

# clean up factor names for display
heatmap_data_full.index = [clean_factor_name(f) for f in heatmap_data_full.index]

# Calculate figure height based on number of factors (~0.25 inches per factor)
n_factors = len(heatmap_data_full)
fig_height = max(40, n_factors * 0.25)

fig, ax = plt.subplots(figsize=(14, fig_height))
hm = sns.heatmap(
    heatmap_data_full,
    cmap='RdBu_r',
    center=0,
    vmin=-3,
    vmax=3,
    xticklabels=True,
    yticklabels=True,
    ax=ax,
    cbar_kws={'label': 'Z-score (normalized across cell types)', 'shrink': 0.3}
)
ax.set_xlabel('Cell Type', fontsize=14, fontweight='bold')
ax.set_ylabel('Global Factor', fontsize=14, fontweight='bold')
ax.set_title('All 151 Global Factors by Cell Type\n(Z-scored activity across cell types)', 
             fontsize=16, fontweight='bold', pad=20)

plt.xticks(rotation=45, ha='right', fontsize=11)
plt.yticks(fontsize=7)  # Small but readable for 151 factors
plt.tight_layout()
plt.show()

2. Clustered heatmap with dendrograms

*Row dendrogram (left side - factors):*
- Groups factors with similar cell-type activation profiles
- **Tight clusters** = factors that behave similarly across all cell types (co-regulated pathways or redundant gene programs)
- Factors in the same cluster may represent functionally related biological processes

*Column dendrogram (top - cell types):*
- Groups cell types with similar factor profiles

**Block Structure:**
- Rectangular regions of similar color = **modules** of factors co-active in subsets of cell types
- These modules often represent lineage-specific biological processes

In [ ]:
fig_height_cluster = max(45, n_factors * 0.28)

# clustermap with all factors
g = sns.clustermap(
    heatmap_data_full,
    cmap='RdBu_r',
    center=0,
    vmin=-3,
    vmax=3,
    figsize=(16, fig_height_cluster),
    dendrogram_ratio=(0.08, 0.12),  # (row_dendrogram, col_dendrogram) proportions
    cbar_pos=(0.02, 0.85, 0.02, 0.1),  # Colorbar position
    xticklabels=True,
    yticklabels=True,
    method='ward',  # Clustering method - ward minimizes variance
    metric='euclidean',  # Distance metric
    row_cluster=True,  # Cluster factors
    col_cluster=True   # Cluster cell types
)
g.ax_heatmap.set_xlabel('Cell Type', fontsize=14, fontweight='bold')
g.ax_heatmap.set_ylabel('Global Factor (clustered)', fontsize=14, fontweight='bold')
g.ax_heatmap.tick_params(axis='y', labelsize=6)  # Smaller for clustered view
g.ax_heatmap.tick_params(axis='x', labelsize=11, rotation=45)
plt.setp(g.ax_heatmap.get_xticklabels(), ha='right', rotation=45)
g.fig.suptitle('Hierarchically Clustered Global Factors by Cell Type\n(Ward linkage, Euclidean distance)', 
               fontsize=16, fontweight='bold', y=1.01)
plt.show()

3. summary stats

**Panel A (Histogram):**
- Shows how many factors are highly cell-type specific vs broadly expressed
- Factors with max z-score > 1.5 are considered "specific" to at least one cell type
- A right-skewed distribution suggests many factors have strong cell-type preferences

**Panel B (Bar chart):**
- Shows which cell types have the most specific factors associated with them
- Cell types with more specific factors may have more unique transcriptional programs
- In cancer, this might highlight cells with distinct functional states

**Panel C (Threshold matrix):**
- Shows how specificity counts change at different thresholds
- Helps you decide on an appropriate threshold for your downstream analysis
- Cell types that maintain high counts at z > 3.0 have very strongly distinguishing factors

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel A: Distribution of max z-scores per factor
max_zscores = celltype_factor_zscore.max(axis=0)
axes[0].hist(max_zscores, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(x=1.5, color='red', linestyle='--', linewidth=2, label='Specificity threshold (1.5)')
axes[0].set_xlabel('Maximum Z-score per Factor', fontsize=12)
axes[0].set_ylabel('Number of Factors', fontsize=12)
axes[0].set_title('Distribution of Factor Specificity\n(higher = more cell-type specific)', 
                  fontsize=12, fontweight='bold')
axes[0].legend()

# Panel B: Number of specific factors per cell type
n_specific = (celltype_factor_zscore > 1.5).sum(axis=1)
bars = axes[1].barh(n_specific.index, n_specific.values, color='coral', edgecolor='white')
axes[1].set_xlabel('Number of Specific Factors (z > 1.5)', fontsize=12)
axes[1].set_ylabel('Cell Type', fontsize=12)
axes[1].set_title('Cell Type Factor Burden\n(factors with z-score > 1.5)', 
                  fontsize=12, fontweight='bold')
for bar, val in zip(bars, n_specific.values):
    axes[1].text(val + 0.5, bar.get_y() + bar.get_height()/2, str(val), 
                 va='center', fontsize=10)

# Panel C: Heatmap of factor specificity counts at different thresholds
specificity_matrix = pd.DataFrame(index=celltype_factor_zscore.index)
for threshold in [1.0, 1.5, 2.0, 2.5, 3.0]:
    specificity_matrix[f'z > {threshold}'] = (celltype_factor_zscore > threshold).sum(axis=1)

sns.heatmap(specificity_matrix, annot=True, fmt='d', cmap='YlOrRd', ax=axes[2],
            cbar_kws={'label': 'Number of factors'})
axes[2].set_xlabel('Z-score Threshold', fontsize=12)
axes[2].set_ylabel('Cell Type', fontsize=12)
axes[2].set_title('Factor Counts by Specificity Threshold', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

8. Identify factors specific to each cell type

In [ ]:
celltype_top_factors = {}

for ct in celltype_factor_zscore.index:
    # Get factors ranked by z-score for this cell type
    ct_scores = celltype_factor_zscore.loc[ct].sort_values(ascending=False)
    
    # Top 5 factors for this cell type
    top_factors = ct_scores.head(5)
    
    celltype_top_factors[ct] = top_factors
    
    print(f"\n{ct}:")
    for factor, zscore in top_factors.items():
        factor_short = clean_factor_name(factor)
        print(f"  {factor_short[:50]}: z={zscore:.2f}")
